In [1]:
import pandas as pd
import numpy as np
import joblib
import dice_ml
from dice_ml import Dice
import warnings
import os
from tqdm import tqdm  # Progress bar (install via: pip install tqdm)

warnings.filterwarnings('ignore')

print("=" * 80)
print("Step 3: DiCE Counterfactual Explanation Generation (Genetic Method - High Performance)")
print("=" * 80)

# ============================================================================
# 1. Data and Model Loading
# ============================================================================
print("\n[Step 1] Loading Data and Model")

# 1-1. Load original dataset
if os.path.exists('selected_data_for_modeling.csv'):
    df_original = pd.read_csv('selected_data_for_modeling.csv')
else:
    raise FileNotFoundError("'selected_data_for_modeling.csv' not found.")

# 1-2. Load model and parameters
# Using filenames saved in Step 2 (base_model_final.pkl)
try:
    model = joblib.load('base_model_final.pkl')
    feature_names = joblib.load('selected_features_final.pkl')
    threshold = joblib.load('base_model_threshold_final.pkl')

    print(f"Model loaded successfully: base_model_final.pkl")
    print(f"Decision threshold: {threshold:.4f}")

except FileNotFoundError:
    raise FileNotFoundError("Model file (base_model_final.pkl) not found. Please run Step 2 first.")

# ============================================================================
# 2. Define Analysis Targets (All Bankrupt Firms)
# ============================================================================
print("\n[Step 2] Defining Analysis Targets")

# Filter only firms with actual bankruptcy (PERF_12M == 1)
target_df = df_original[df_original['PERF_12M'] == 1].copy()
X_target = target_df[feature_names]
target_ids = target_df['ID'].values

print(f"Analysis targets: {len(X_target)} firms (all bankrupt cases)")

# ============================================================================
# 3. DiCE Object Initialisation (Genetic Algorithm)
# ============================================================================
print("\n[Step 3] Initialising DiCE Object (Genetic Algorithm)")

# Data object
d = dice_ml.Data(
    dataframe=df_original.drop('ID', axis=1),
    continuous_features=feature_names,
    outcome_name='PERF_12M'
)

# Model object
m = dice_ml.Model(model=model, backend='sklearn', model_type='classifier')

# Create Explainer — key setting: method='genetic'
# The Genetic Algorithm uses an evolutionary process to find optimal solutions.
# It takes longer than Random search but virtually eliminates "No Counterfactuals found" errors.
exp = Dice(d, m, method='genetic')

print("Configuration complete: method='genetic' (optimised for success rate)")

# ============================================================================
# 4. Counterfactual (CF) Generation Loop
# ============================================================================
print("\n[Step 4] Starting Counterfactual Generation")
print("Note: The Genetic method performs precise computation and may take some time.")

cf_results_list = []
cf_failed_ids = []

# Display progress with tqdm
for i in tqdm(range(len(X_target)), desc="Processing"):
    try:
        row = X_target.iloc[i]
        current_id = target_ids[i]

        # Format input for DiCE
        query_instance = row.to_frame().T

        # Compute original bankruptcy probability
        original_prob = model.predict_proba(query_instance)[0, 1]

        # -----------------------------------------------------------
        # [Key Design] Define Immutable Features
        # -----------------------------------------------------------
        # Variables with '_1' suffix represent prior-year (lagged) figures
        # that cannot be retroactively altered by any managerial action.
        immutable_features = [
            'FN1_11_1', 'FN1_13_1', 'FN2_1_1', 'FN2_3_2', 'FN2_5_1',
            'FN2_10_1', 'FN3_11_1', 'FN1_24_1'
            # Include all variables from feature_names that represent
            # prior-year or historical financial figures.
        ]

        # Allow modification only for non-immutable features
        features_to_vary = [f for f in feature_names if f not in immutable_features]

        # -----------------------------------------------------------
        # Generate CFs
        # -----------------------------------------------------------
        dice_exp = exp.generate_counterfactuals(
            query_instance,
            total_CFs=4,
            desired_class=0,
            features_to_vary=features_to_vary,  # Only present-period variables
            verbose=False,
        )

        cf_df = dice_exp.cf_examples_list[0].final_cfs_df

        if cf_df is not None and not cf_df.empty:
            # Save all 4 generated CFs
            for cf_idx, cf_row in cf_df.iterrows():
                cf_features = cf_row[feature_names].values.reshape(1, -1)
                cf_prob = model.predict_proba(cf_features)[0, 1]

                result_entry = {
                    'ID': current_id,
                    'CF_Number': cf_idx + 1,
                    'Original_Proba': original_prob,
                    'Target_Proba': cf_prob
                }

                # Save original, CF, and delta values per feature
                for feat in feature_names:
                    orig_val = row[feat]
                    cf_val = cf_row[feat]
                    result_entry[f'Original_{feat}'] = orig_val
                    result_entry[f'CF_{feat}'] = cf_val
                    result_entry[f'Change_{feat}'] = cf_val - orig_val

                cf_results_list.append(result_entry)
        else:
            cf_failed_ids.append(current_id)

    except Exception as e:
        # Log failure and continue
        cf_failed_ids.append(current_id)
        continue

# ============================================================================
# 5. Save Results
# ============================================================================
print("\n" + "=" * 80)
print("[Step 5] Saving Results")
print("-" * 80)

final_cf_df = pd.DataFrame(cf_results_list)

# Summary statistics
total_target = len(X_target)
success_cnt = final_cf_df['ID'].nunique() if not final_cf_df.empty else 0
fail_cnt = len(cf_failed_ids)

print(f"Total analysis targets: {total_target}")
print(f"Successful firms: {success_cnt} (success rate: {success_cnt/total_target*100:.1f}%)")
print(f"Failed firms: {fail_cnt}")

if not final_cf_df.empty:
    output_filename = 'cf_results_full.csv'
    final_cf_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\nResults saved: {output_filename}")

    # Sample output
    print("\n[Sample Output (First Successful Firm)]")
    sample = final_cf_df.iloc[0]
    print(f"ID: {sample['ID']}")
    print(f"Bankruptcy probability change: {sample['Original_Proba']:.4f} -> {sample['Target_Proba']:.4f}")
else:
    print("\n[Warning] No CFs generated. Please check the model and data.")

if cf_failed_ids:
    pd.DataFrame({'ID': cf_failed_ids}).to_csv('cf_failed_ids.csv', index=False)
    print("Failed ID list saved: cf_failed_ids.csv")

print("\nStep 3 complete.")

Step 3: DiCE Counterfactual Explanation Generation (Genetic Method - High Performance)

[Step 1] Loading Data and Model
Model loaded successfully: base_model_final.pkl
Decision threshold: 0.6989

[Step 2] Defining Analysis Targets
Analysis targets: 266 firms (all bankrupt cases)

[Step 3] Initialising DiCE Object (Genetic Algorithm)
Configuration complete: method='genetic' (optimised for success rate)

[Step 4] Starting Counterfactual Generation
Note: The Genetic method performs precise computation and may take some time.


Processing: 100%|████████████████████████████████████████████████████████████████████| 266/266 [01:53<00:00,  2.34it/s]


[Step 5] Saving Results
--------------------------------------------------------------------------------
Total analysis targets: 266
Successful firms: 266 (success rate: 100.0%)
Failed firms: 0

Results saved: cf_results_full.csv

[Sample Output (First Successful Firm)]
ID: 51029.0
Bankruptcy probability change: 0.9953 -> 0.2762

Step 3 complete.
